# 3. Local Meeting Notes Summarizer

## What We're Building
A tool that takes raw meeting transcripts or notes and produces:
- **Executive summary** (2-3 sentences)
- **Key decisions** made
- **Action items** with owners
- **Open questions / blockers**

**You will learn:**
- Prompt engineering for structured extraction
- Chain composition with LangChain
- Structured output formatting
- Handling long documents with map-reduce

**Stack:** Ollama, LangChain

In [ ]:
# !pip install -q langchain langchain-ollama

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize local LLM
llm = ChatOllama(model="qwen3:8b", temperature=0.1)
print("LLM ready!")

## Step 1 — Sample Meeting Transcript

In [ ]:
meeting_transcript = """
Meeting: Q3 Product Planning
Date: 2024-11-15
Attendees: Sarah (PM), Alex (Eng Lead), Jordan (Design), Casey (QA)

Sarah: Let's review the Q3 priorities. We need to finalize the feature list.

Alex: We've been looking at the performance issues. The API latency is up 40% since
last month. I think we need to prioritize the database optimization before adding
new features.

Sarah: Agreed. Can you estimate the effort for the DB work?

Alex: About 2-3 sprints. We'd need to migrate to connection pooling and add indexing.
Mike can lead that.

Jordan: For the redesign, the new dashboard mockups are ready. I'll share the Figma
link after this meeting. We need feedback by Friday.

Casey: I found 3 critical bugs in the payment flow. Two are related to the timeout
handling. We should fix those before the next release.

Sarah: Definitely. Casey, can you file those as P0 tickets?

Casey: Already done. Tickets are PROD-441, PROD-442, and PROD-443.

Alex: One concern — we don't have enough test coverage for the payment module.
Current coverage is only 45%. We should get that to at least 80% before any
major changes.

Sarah: Good point. Let's add that as a Q3 goal. Jordan, when can you have the
final designs for the settings page?

Jordan: End of next week. I'm waiting on the accessibility audit results.

Sarah: OK, let's reconvene next Tuesday with updated estimates. Alex, please
prepare a tech debt assessment. Jordan, share the mockups today.

Meeting ended at 11:45 AM.
"""

print(f"Transcript length: {len(meeting_transcript)} characters")
print("Preview:")
print(meeting_transcript[:200])

## Step 2 — Build the Summarization Chain

In [ ]:
# Define the summarization prompt
summarize_prompt = ChatPromptTemplate.from_template(
    """You are a meeting notes assistant. Analyze the following meeting transcript
and produce a structured summary.

MEETING TRANSCRIPT:
{transcript}

Produce the following sections:

## Executive Summary
(2-3 sentence overview of what was discussed and decided)

## Key Decisions
(Bullet list of decisions made)

## Action Items
(Bullet list with format: "- [ ] [Owner]: [Task] — [Deadline if mentioned]")

## Open Questions / Blockers
(Anything unresolved or blocking progress)

## Follow-up Meeting
(Next meeting details if mentioned)

Be specific. Use names and ticket numbers from the transcript."""
)

# Build the chain
summary_chain = summarize_prompt | llm | StrOutputParser()

# Run it
summary = summary_chain.invoke({"transcript": meeting_transcript})
print(summary)

## Step 3 — Extract Structured Action Items

In [ ]:
from langchain.prompts import ChatPromptTemplate

# Separate chain just for action items extraction
action_prompt = ChatPromptTemplate.from_template(
    """Extract all action items from this meeting transcript.
For each action item, provide:
1. Owner (who is responsible)
2. Task (what needs to be done)
3. Deadline (if mentioned, otherwise "TBD")
4. Priority (High/Medium/Low based on context)

TRANSCRIPT:
{transcript}

Format as a numbered list:
1. **Owner**: Task — Deadline — Priority"""
)

action_chain = action_prompt | llm | StrOutputParser()
actions = action_chain.invoke({"transcript": meeting_transcript})
print("=== Extracted Action Items ===")
print(actions)

## Step 4 — Handle Long Transcripts with Map-Reduce

For meetings longer than the LLM context window, we split and summarize
each part, then combine summaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def summarize_long_transcript(transcript: str, chunk_size: int = 2000) -> str:
    """Summarize long transcripts using a map-reduce approach."""
    # Split into manageable chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=200,
    )
    chunks = splitter.split_text(transcript)

    if len(chunks) == 1:
        # Short enough for single pass
        return summary_chain.invoke({"transcript": transcript})

    # Map phase: summarize each chunk
    map_prompt = ChatPromptTemplate.from_template(
        """Summarize the key points, decisions, and action items from this
portion of a meeting transcript:

{chunk}

Provide a concise bullet-point summary."""
    )
    map_chain = map_prompt | llm | StrOutputParser()

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        print(f"  Processing chunk {i+1}/{len(chunks)}...")
        s = map_chain.invoke({"chunk": chunk})
        chunk_summaries.append(s)

    # Reduce phase: combine all chunk summaries
    combined = "\n\n---\n\n".join(chunk_summaries)
    reduce_prompt = ChatPromptTemplate.from_template(
        """Below are summaries from different parts of the same meeting.
Combine them into one cohesive meeting summary with these sections:

## Executive Summary
## Key Decisions
## Action Items
## Open Questions / Blockers

PARTIAL SUMMARIES:
{summaries}"""
    )
    reduce_chain = reduce_prompt | llm | StrOutputParser()
    final = reduce_chain.invoke({"summaries": combined})
    return final

# Test with our transcript (will use single-pass since it's short)
result = summarize_long_transcript(meeting_transcript)
print(result)

In [ ]:
# Reusable function for any meeting
def process_meeting(transcript: str, verbose: bool = True):
    """Process a meeting transcript and return structured notes."""
    summary = summary_chain.invoke({"transcript": transcript})
    actions = action_chain.invoke({"transcript": transcript})

    if verbose:
        print("=" * 60)
        print("MEETING SUMMARY")
        print("=" * 60)
        print(summary)
        print("\n" + "=" * 60)
        print("ACTION ITEMS")
        print("=" * 60)
        print(actions)

    return {"summary": summary, "actions": actions}

# Run it
result = process_meeting(meeting_transcript)

## Summary & Next Steps

**What you learned:**
- ✅ Prompt engineering for structured text extraction
- ✅ LCEL chain composition (prompt | llm | parser)
- ✅ Map-reduce pattern for long documents
- ✅ Extracting multiple output types from the same input

**Next steps:**
- Connect to a real transcript source (Whisper output, Zoom export)
- Add Pydantic output parsing for machine-readable action items
- Build a recurring meeting tracker
- Try different models for speed vs quality tradeoff

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
